# LLM Agent Orchestration with LangChain and Granite

This notebook demonstrates how to build an LLM agent orchestration pipeline using IBM® watsonx.ai Granite model and LangChain. The pipeline processes a text document (The Adventures of Sherlock Holmes), creates a vector index for semantic search, and answers user queries using Retrieval-Augmented Generation (RAG).

**Reference:** https://www.ibm.com/es-es/think/tutorials/llm-agent-orchestration-with-langchain-and-granite

## Step 3. Install Required Packages

Install the essential libraries needed to work with LangChain and IBM® watsonx.ai.

In [ ]:
!pip install --upgrade pip

!pip install langchain faiss-cpu pandas sentence-transformers

%pip install langchain

!pip install langchain-ibm

### Package descriptions

- **`langchain`**: The core framework for building applications with language models.
- **`faiss-cpu`**: For efficient similarity search, used to create and query vector indexes.
- **`pandas`**: For data manipulation and analysis.
- **`sentence-transformers`**: To generate embeddings for semantic search.
- **`langchain-ibm`**: To integrate IBM watsonxLLM (`ibm/granite-3-8b-instruct`) with LangChain.

## Step 4. Import Necessary Libraries

In [ ]:
import os

from langchain_ibm import WatsonxLLM

from langchain.embeddings import HuggingFaceEmbeddings

from langchain.vectorstores import FAISS

from langchain.text_splitter import RecursiveCharacterTextSplitter

import pandas as pd

import getpass

### Module descriptions

- **`os`**: Provides a way to interact with the operating system (e.g., accessing environment variables).
- **`langchain_ibm.WatsonxLLM`**: Allows using IBM® watsonx Granite LLM seamlessly within the LangChain framework.
- **`langchain.embeddings.HuggingFaceEmbeddings`**: Generates embeddings for text using HuggingFace models, essential for semantic search.
- **`langchain.vectorstores.FAISS`**: A library for efficient vector storage and similarity search.
- **`RecursiveCharacterTextSplitter`**: Helps split large blocks of text into smaller chunks.
- **`pandas`**: A powerful library for data analysis and manipulation.
- **`getpass`**: A secure way to capture sensitive input (e.g., API keys) without displaying them on screen.

## Step 5. Configure Credentials

Set up credentials to access the IBM® watsonx Machine Learning (WML) API.

In [ ]:
# Set up credentials
credentials = {
    "url": "https://us-south.ml.cloud.ibm.com",  # Replace with the correct region if needed
    "apikey": getpass.getpass("Please enter your WML API key (hit enter): ")
}

# Set up project_id
try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id (hit enter): ")

## Step 6. Initialize the Large Language Model

Initialize IBM watsonxLLM using the `ibm/granite-3-8b-instruct` model.

In [ ]:
# Initialize the IBM Granite LLM
llm = WatsonxLLM(
    model_id="ibm/granite-3-8b-instruct",
    url=credentials["url"],
    apikey=credentials["apikey"],
    project_id=project_id,
    params={
        "max_new_tokens": 150
    }
)

## Step 7. Define a Function to Extract Text from a File

This function reads and extracts the content of a plain text file.

In [ ]:
def extract_text_from_txt(file_path):
    """Extracts text from a plain text file."""
    with open(file_path, "r", encoding="utf-8") as file:
        text = file.read()
    return text

## Step 8. Split the Text into Chunks

Split large blocks of text into smaller, manageable chunks for efficient processing and indexing.

In [ ]:
def split_text_into_chunks(text, chunk_size=500, chunk_overlap=50):
    """Splits text into smaller chunks for indexing."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_text(text)

## Step 9. Create a Vector Index

Convert text chunks into embeddings and store them in a FAISS vector index for semantic search.

In [ ]:
def create_vector_index(chunks):
    """Creates a FAISS vector index from text chunks."""
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.from_texts(chunks, embeddings)
    return vector_store

## Step 10. Query the Vector Index with Granite

Query the vector index to retrieve relevant information and use IBM Granite LLM to generate a refined response.

In [ ]:
def query_index_with_granite_dynamic(vector_store, query, llm):
    """Searches the vector index, uses Granite to refine the response, and returns all components."""
    # Perform similarity search
    print("\n> Entering new AgentExecutor chain...")
    thought = f"The query '{query}' requires context from the book to provide an accurate response."
    print(f" Thought: {thought}")
    action = "Search FAISS Vector Store"
    print(f" Action: {action}")
    action_input = query
    print(f" Action Input: \"{action_input}\"")

    # Retrieve context
    results = vector_store.similarity_search(query, k=3)
    observation = "\n".join([result.page_content for result in results])
    print(f" Observation:\n{observation}\n")

    # Generate response with Granite
    prompt = f"Context:\n{observation}\n\nQuestion: {query}\nAnswer:"
    print(f" Thought: Combining retrieved context with the query to generate a detailed answer.")
    final_answer = llm.invoke(prompt)
    print(f" Final Answer: {final_answer.strip()}")
    print("\n> Finished chain.")

    # Return all components as a dictionary
    return {
        "Thought": thought,
        "Action": action,
        "Action Input": action_input,
        "Observation": observation,
        "Final Answer": final_answer.strip()
    }

## Step 11. Generate a DataFrame for Query Results

Process multiple queries dynamically, retrieve relevant information, and save the results in a structured format.

In [ ]:
def dynamic_output_to_dataframe(vector_store, queries, llm, csv_filename="output.csv"):
    """Generates a DataFrame dynamically for multiple queries and saves it as a CSV file."""
    # List to store all query outputs
    output_data = []

    # Process each query
    for query in queries:
        # Capture the output dynamically
        output = query_index_with_granite_dynamic(vector_store, query, llm)
        output_data.append(output)

    # Convert the list of dictionaries into a DataFrame
    df = pd.DataFrame(output_data)

    # Display the DataFrame
    print("\nFinal DataFrame:")
    print(df)

    # Save the DataFrame as a CSV file
    df.to_csv(csv_filename, index=False)
    print(f"\nOutput saved to {csv_filename}")

## Step 12. Run the Main Workflow

Combine all previous steps into a single workflow to process the text file, answer user queries, and save results in a structured format.

> **Note:** Make sure the file `aosh.txt` (The Adventures of Sherlock Holmes) is available in the same directory as this notebook. You can download the text from [Project Gutenberg](https://www.gutenberg.org/ebooks/1661).

In [ ]:
def main_workflow():
    # Replace with your text file
    file_path = "aosh.txt"

    # Extract text from the text file
    text = extract_text_from_txt(file_path)

    # Split the text into chunks
    chunks = split_text_into_chunks(text)

    # Create a vector index
    vector_store = create_vector_index(chunks)

    # Define queries
    queries = [
        "What is the plot of 'A Scandal in Bohemia'?",
        "Who is Dr. Watson, and what role does he play in the stories?",
        "Describe the relationship between Sherlock Holmes and Irene Adler.",
        "What methods does Sherlock Holmes use to solve cases?"
    ]

    # Generate and save output dynamically
    dynamic_output_to_dataframe(vector_store, queries, llm)


# Run the workflow
main_workflow()

## Sección adicional: Tool Calling local con Ollama

En esta sección usaremos `ollama` para que el modelo invoque herramientas que inspeccionan archivos locales.

Objetivo:
- Buscar información en archivos `.txt` y `.pdf`.
- Buscar imágenes relacionadas en `.jpg`, `.jpeg` y `.png`.
- Devolver una respuesta basada en los resultados de herramientas.

> Requisitos previos:
> - Tener Ollama instalado (`ollama -v`).
> - Haber descargado modelos compatibles con tools/vision.

In [ ]:
# Instalar dependencias de esta sección (si hace falta)
# %pip install ollama pymupdf

# Descargar modelos (ejecutar una sola vez)
# !ollama pull granite3.2:8b
# !ollama pull granite3.2-vision

In [ ]:
import os
import subprocess
from pathlib import Path

import ollama
import pymupdf

TEXT_MODEL = "granite3.2:8b"
VISION_MODEL = "granite3.2-vision"
FILES_DIR = Path("./files")


def search_text_files(keyword: str, files_dir: Path = FILES_DIR) -> str:
    """Busca keyword en .txt y .pdf locales con apoyo del LLM."""
    if not files_dir.exists():
        return "None"

    for entry in files_dir.iterdir():
        if not entry.is_file() or entry.name.startswith("."):
            continue

        if entry.suffix.lower() == ".pdf":
            document_text = ""
            doc = pymupdf.open(entry)
            for page in doc:
                document_text += page.get_text()
            doc.close()

            prompt = (
                "Respond only 'yes' or 'no', do not add any additional information. "
                f"Is the following text about {keyword}? {document_text}"
            )
            res = ollama.chat(model=TEXT_MODEL, messages=[{"role": "user", "content": prompt}])
            if "yes" in res["message"]["content"].lower():
                return str(entry)

        elif entry.suffix.lower() == ".txt":
            file_content = entry.read_text(encoding="utf-8", errors="ignore")
            prompt = (
                "Respond only 'yes' or 'no', do not add any additional information. "
                f"Is the following text about {keyword}? {file_content}"
            )
            res = ollama.chat(model=TEXT_MODEL, messages=[{"role": "user", "content": prompt}])
            if "yes" in res["message"]["content"].lower():
                return str(entry)

    return "None"


def search_image_files(keyword: str, files_dir: Path = FILES_DIR) -> str:
    """Busca keyword en descripciones de imagenes con modelo vision."""
    if not files_dir.exists():
        return "None"

    image_ext = {".jpg", ".jpeg", ".png"}
    for entry in files_dir.iterdir():
        if not entry.is_file() or entry.name.startswith("."):
            continue
        if entry.suffix.lower() not in image_ext:
            continue

        res = ollama.chat(
            model=VISION_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": "Describe this image in short sentences. Use simple phrases first and then describe it more fully.",
                    "images": [str(entry)],
                }
            ],
        )
        if keyword.lower() in res["message"]["content"].lower():
            return str(entry)

    return "None"


available_functions = {
    "Search inside text files": search_text_files,
    "Search inside image files": search_image_files,
}

ollama_tools = [
    {
        "type": "function",
        "function": {
            "name": "Search inside text files",
            "description": "This tool searches in PDF or plaintext files in the local file system for descriptions or mentions of the keyword.",
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {
                        "type": "string",
                        "description": "Generate one keyword from the user request to search for in text files",
                    }
                },
                "required": ["keyword"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "Search inside image files",
            "description": "This tool searches for photos or image files in the local file system for the keyword.",
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {
                        "type": "string",
                        "description": "Generate one keyword from the user request to search for in image files",
                    }
                },
                "required": ["keyword"],
            },
        },
    },
]

print("Herramientas listas. Asegurate de tener archivos en ./files")

In [ ]:
# Iniciar ollama (opcional si ya esta corriendo)
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

# Modo demo no interactivo (recomendado para pruebas repetibles)
NON_INTERACTIVE = True
DEMO_QUERY = "informacion sobre perros"

if NON_INTERACTIVE:
    user_input = DEMO_QUERY
else:
    user_input = input("What would you like to search for? ").strip()

print("Consulta:", user_input)

messages = [{"role": "user", "content": user_input}]

response = ollama.chat(
    model=TEXT_MODEL,
    messages=messages,
    tools=ollama_tools,
)

output = []

if response.message.tool_calls:
    for tool_call in response.message.tool_calls:
        function_to_call = available_functions.get(tool_call.function.name)
        if function_to_call:
            print("Calling tool:", tool_call.function.name, "with arguments:", tool_call.function.arguments)
            tool_res = function_to_call(**tool_call.function.arguments)
            print("Tool response is", str(tool_res))
            if str(tool_res) != "None":
                output.append(str(tool_res))
        else:
            print("Could not find", tool_call.function.name)

    messages.append(response.message)

    prompt = (
        "If the tool output contains one or more file names, then give the user only the filename found. "
        "Do not add additional details. If the tool output is empty ask the user to try again. "
        "Here is the tool output: "
    )

    messages.append({"role": "tool", "content": prompt + " " + ", ".join(output)})

    final_response = ollama.chat(model=TEXT_MODEL, messages=messages)
    print("Final response:", final_response.message.content)
else:
    print("No tool calls returned from model")

## Sección adicional: Tool Calling con LM Studio

En esta sección tienes dos rutas de ejecución:

- Ruta rápida: cálculo + conteo de letras con herramientas.
- Ruta completa: agente de ajedrez con tool calling.

Sigue los pasos en orden y detente en la ruta que necesites.

### Paso 1. Instalar LM Studio

1. Verifica requisitos mínimos de tu sistema.
2. Descarga LM Studio desde: https://lmstudio.ai/
3. Carga un modelo local (ejemplo: `ibm-granite/granite-3.3-8b-instruct-GGUF`).
4. Inicia el servidor local desde **Developer** y cambia **Status** a **Running**.

> Nota: esta parte se realiza fuera del notebook.

### Paso 2. Instalar dependencias

In [ ]:
# Instala dependencias para LM Studio tool calling (ejecutar una sola vez)
# %pip install git+https://github.com/ibm-granite-community/utils lmstudio chess

import lmstudio as lms

### Paso 3. Cargar el modelo y probar una respuesta simple

In [ ]:
MODEL_NAME = "ibm-granite/granite-3.3-8b-instruct-GGUF"
model = lms.llm(MODEL_NAME)

print(model.respond("Hello Granite!"))

### Ruta rápida (recomendada para demo): Paso 4

Prueba de cálculo sin herramientas.

In [ ]:
print(model.respond("What is 26.97 divided by 6.28? Don't round."))

### Paso 5. Herramientas matemáticas en Python y tool calling automático

In [ ]:
def add(a: float, b: float) -> float:
    """Given two numbers a and b, return a + b."""
    return a + b


def subtract(a: float, b: float) -> float:
    """Given two numbers a and b, return a - b."""
    return a - b


def multiply(a: float, b: float) -> float:
    """Given two numbers a and b, return a * b."""
    return a * b


def divide(a: float, b: float) -> float:
    """Given two numbers a and b, return a / b."""
    return a / b


def exp(a: float, b: float) -> float:
    """Given two numbers a and b, return a^b."""
    return a ** b


model.act(
    "What is 26.97 divided by 6.28? Don't round.",
    [add, subtract, multiply, divide, exp],
    on_message=print,
)

### Paso 5b. Conteo de letras con herramienta dedicada

In [ ]:
print(model.respond("How many Bs are in the word 'blackberry'?"))


def get_letter_frequency(word: str) -> dict:
    """Retorna la frecuencia de letras en una palabra."""
    letter_frequencies: dict[str, int] = {}
    for letter in word:
        letter_frequencies[letter] = letter_frequencies.get(letter, 0) + 1
    return letter_frequencies


model.act(
    "How many Bs are in the word 'blackberry'?",
    [get_letter_frequency],
    on_message=print,
)

### Ruta completa: Paso 6 (agente de ajedrez con tool calling)

Esta parte es más larga e interactiva (usuario vs IA).

In [ ]:
import os
import random
import requests
import shutil

import chess
import chess.polyglot
import chess.svg
from IPython.display import SVG, clear_output, display

board = chess.Board()
ai_pos = 0

RAW_URL = (
    "https://raw.githubusercontent.com/"
    "niklasf/python-chess/master/data/polyglot/performance.bin"
)
DEST_FILE = "performance.bin"

if not os.path.exists(DEST_FILE):
    print("Downloading performance.bin ...")
    with requests.get(RAW_URL, stream=True, timeout=15) as r:
        r.raise_for_status()
        with open(DEST_FILE, "wb") as out:
            shutil.copyfileobj(r.raw, out, 1 << 16)


def legal_moves() -> list[str]:
    return [board.san(move) for move in board.legal_moves]


def possible_captures() -> list[dict]:
    result = []
    for move in board.generate_legal_captures():
        piece = board.piece_at(move.to_square)
        piece_type = piece.symbol().upper() if piece else "?"
        defenders = board.attackers(not board.turn, move.to_square)
        is_hanging = len(defenders) == 0
        result.append(
            {
                "san": board.san(move),
                "captured_piece": piece_type,
                "is_hanging": is_hanging,
            }
        )
    return result


def possible_checks() -> list[dict]:
    result = []
    for move in board.legal_moves:
        if not board.gives_check(move):
            continue
        temp = board.copy()
        temp.push(move)

        can_capture = any(
            temp.is_capture(reply) and reply.to_square == move.to_square
            for reply in temp.legal_moves
        )

        king_sq = temp.king(not board.turn)
        can_escape = any(reply.from_square == king_sq for reply in temp.legal_moves)

        can_block = any(
            (not temp.is_capture(reply))
            and reply.from_square != king_sq
            and (not temp.gives_check(reply))
            for reply in temp.legal_moves
        )

        result.append(
            {
                "san": board.san(move),
                "can_be_captured": can_capture,
                "can_be_blocked": can_block,
                "can_escape_by_moving_king": can_escape,
            }
        )
    return result


def get_move_history() -> list[str]:
    return [board.san(move) for move in board.move_stack]


def get_book_moves() -> list[str]:
    moves = []
    with chess.polyglot.open_reader("performance.bin") as reader:
        for entry in reader.find_all(board):
            moves.append(board.san(entry.move))
    return moves


def is_ai_turn() -> bool:
    return bool(board.turn) == (ai_pos == 0)


def make_ai_move(move: str) -> None:
    if is_ai_turn():
        board.push_san(move)
    else:
        raise ValueError("Not AI's turn.")


def make_user_move(move: str) -> None:
    if not is_ai_turn():
        board.push_san(move)
    else:
        raise ValueError("Not player's turn.")

In [ ]:
chat_white = lms.Chat(
    """You are a chess AI, playing for white. Your task is to make the best move in the current position, using the provided tools.
Use your chess knowledge (openings, tactics, strategy) and use tools to validate board state.
Always prefer book moves when available. Be careful with checks and captures."""
)

chat_black = lms.Chat(
    """You are a chess AI, playing for black. Your task is to make the best move in the current position, using the provided tools.
Use your chess knowledge (openings, tactics, strategy) and use tools to validate board state.
Always prefer book moves when available. Be careful with checks and captures."""
)

move = 0
board.reset()
ai_pos = round(random.random())


def update_board(move_num: int = 0, ai_side: int = 0) -> None:
    clear_output(wait=True)
    print(f"Board after move {move_num + 1}")
    if ai_side == 1:
        display(SVG(chess.svg.board(board, size=420)))
    else:
        display(SVG(chess.svg.board(board, size=420, orientation=chess.BLACK)))


def get_end_state() -> str | None:
    if board.is_checkmate():
        return "Checkmate!"
    if board.is_stalemate():
        return "Stalemate!"
    if board.is_insufficient_material():
        return "Draw by insufficient material!"
    if board.is_seventyfive_moves():
        return "Draw by 75-move rule!"
    if board.is_fivefold_repetition():
        return "Draw by fivefold repetition!"
    return None


update_board(move, ai_pos)
print("Escribe 'help' para ver movimientos legales o 'quit' para salir.")

In [ ]:
# Selector de ruta para esta celda:
# True  -> demo rapida (omite ajedrez)
# False -> ejecuta partida completa usuario vs IA
RUN_FAST_DEMO = True

if RUN_FAST_DEMO:
    print("RUN_FAST_DEMO=True: se omite la ruta completa de ajedrez.")
    print("Para jugar contra la IA, cambia RUN_FAST_DEMO a False y vuelve a ejecutar esta celda.")
else:
    user_end_game = False

    while True:
        if ai_pos == 0:
            model.act(
                chat_white,
                [
                    get_move_history,
                    legal_moves,
                    possible_captures,
                    possible_checks,
                    get_book_moves,
                    make_ai_move,
                ],
                on_message=print,
                max_prediction_rounds=8,
            )

            if is_ai_turn():
                make_ai_move(legal_moves()[0])

            update_board(move, ai_pos)
            move += 1
            end_state = get_end_state()
            if end_state:
                print(end_state)
                break

            while True:
                user_move = input(
                    "User (Playing Black): move / help / quit -> "
                ).strip()
                if user_move.lower() == "quit":
                    print("Game ended by user.")
                    user_end_game = True
                    break
                if user_move.lower() == "help":
                    print("Possible moves:", legal_moves())
                    continue
                try:
                    make_user_move(user_move)
                    break
                except ValueError as e:
                    print(e)

            if user_end_game:
                break

            update_board(move, ai_pos)
            move += 1
            end_state = get_end_state()
            if end_state:
                print(end_state)
                break

        else:
            while True:
                user_move = input(
                    "User (Playing White): move / help / quit -> "
                ).strip()
                if user_move.lower() == "quit":
                    print("Game ended by user.")
                    user_end_game = True
                    break
                if user_move.lower() == "help":
                    print("Possible moves:", legal_moves())
                    continue
                try:
                    make_user_move(user_move)
                    break
                except ValueError as e:
                    print(e)

            if user_end_game:
                break

            update_board(move, ai_pos)
            move += 1
            end_state = get_end_state()
            if end_state:
                print(end_state)
                break

            model.act(
                chat_black,
                [
                    get_move_history,
                    legal_moves,
                    possible_captures,
                    possible_checks,
                    get_book_moves,
                    make_ai_move,
                ],
                on_message=print,
                max_prediction_rounds=8,
            )

            if is_ai_turn():
                make_ai_move(legal_moves()[0])

            update_board(move, ai_pos)
            move += 1
            end_state = get_end_state()
            if end_state:
                print(end_state)
                break

### Guía de ejecución rápida

Si solo quieres validar tool calling básico en LM Studio, ejecuta hasta la parte de conteo de letras y termina ahí.

Orden sugerido:
1. Paso 2 (dependencias)
2. Paso 3 (cargar modelo)
3. Ruta rápida: Paso 4, Paso 5 y Paso 5b

Para el agente de ajedrez, continúa con Ruta completa: Paso 6.

## Sección adicional: Resolución de problemas con sistemas multiagente (crewAI)

En esta sección ejecutarás un flujo multiagente para análisis de call center con:

- Transcript Analyzer
- Quality Assurance Specialist
- Report Generator

El proyecto ya está creado en `multiagent-collab-cs-call-center-analysis`.

In [ ]:
# Setup (ejecutar una sola vez)
# %pip install -r multiagent-collab-cs-call-center-analysis/requirements-crewai.txt

from pathlib import Path
import os

BASE = Path("multiagent-collab-cs-call-center-analysis")
assert BASE.exists(), "No existe la carpeta multiagent-collab-cs-call-center-analysis"
print("Proyecto encontrado en:", BASE.resolve())

In [ ]:
# Cargar variables desde .env (raiz del repo) y validar credenciales
from pathlib import Path

ENV_PATH = Path(".env")

if ENV_PATH.exists():
    for raw_line in ENV_PATH.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and value and key not in os.environ:
            os.environ[key] = value
    print("Variables cargadas desde .env")
else:
    print("No se encontro .env en la raiz del repositorio")

required_env = ["WATSONX_APIKEY", "WATSONX_PROJECT_ID", "WATSONX_URL", "SERPER_API_KEY"]
missing = [k for k in required_env if not os.getenv(k)]

if missing:
    print("Faltan variables:", missing)
    print("Completa .env o exportalas manualmente antes de ejecutar el crew.")
else:
    print("Credenciales detectadas.")

In [ ]:
# Ejecutar el sistema multiagente
# Requiere variables de entorno configuradas y dependencias instaladas.

import subprocess

result = subprocess.run(
    ["python", "main.py"],
    cwd=str(BASE),
    capture_output=True,
    text=True,
)

print("Return code:", result.returncode)
print("\n--- STDOUT ---\n")
print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---\n")
    print(result.stderr)

In [ ]:
# Mostrar report.md generado
report_path = BASE / "report.md"

if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))
else:
    print("Aun no existe report.md. Ejecuta la celda anterior primero.")

### Notas

- Si el `return code` es distinto de 0, revisa `STDERR` para identificar credenciales o dependencias faltantes.
- Puedes editar `agents.yaml` y `tasks.yaml` para adaptar roles y objetivos.
- El transcript de ejemplo está en `multiagent-collab-cs-call-center-analysis/data/transcript.txt`.

## Sección adicional: Varias estrategias de colaboración

### Estrategias de colaboración entre agentes

1. Colaboración basada en reglas
- Los agentes siguen reglas explícitas y predefinidas.
- Funciona bien en procesos estables y estructurados.
- Ventaja: alta consistencia.
- Riesgo: baja adaptabilidad ante cambios.

2. Colaboración basada en roles
- Cada agente tiene una función especializada.
- Los resultados se comparten entre etapas para construir el objetivo final.
- Ventaja: modularidad y especialización.
- Riesgo: dependencia fuerte de la integración entre agentes.

3. Colaboración basada en modelos
- Los agentes razonan con modelos del entorno y de otros agentes.
- Útil para incertidumbre y contextos dinámicos.
- Ventaja: flexibilidad de decisión.
- Riesgo: mayor complejidad y coste computacional.

### Marcos más usados

1. IBM Bee Agent Framework
- Diseño modular para sistemas multiagente de nivel productivo.

2. LangChain Agents
- Integración amplia de herramientas y flujos de razonamiento de varios pasos.

3. OpenAI Swarm
- Coordinación ligera por handoffs entre agentes especializados.

### Soluciones empresariales

Watsonx Orchestrate combina:
- habilidades/agentes registrados,
- análisis de intención,
- orquestación de flujos (secuencial/paralelo/reintentos),
- memoria y contexto compartido,
- soporte LLM,
- supervisión humana.

### Predicciones futuras

- Mayor inteligencia colectiva emergente en equipos de agentes.
- Evaluación continua con métricas de precisión, relevancia, eficiencia y explicabilidad.
- Uso creciente en automatización de flujos complejos y decisiones en múltiples etapas.

### Tabla comparativa de estrategias de colaboración

| Estrategia | Cómo funciona | Casos de uso ideales | Complejidad | Coste computacional | Ventaja principal | Riesgo principal |
|---|---|---|---|---|---|---|
| Basada en reglas | Reglas explícitas (if-then, estados, lógica fija) | Procesos estables, cumplimiento normativo, flujos repetitivos | Baja | Bajo | Consistencia y control | Poca adaptabilidad |
| Basada en roles | Agentes especializados por responsabilidad con intercambio de contexto | Pipelines modulares, equipos expertos, análisis por etapas | Media | Medio | Modularidad y especialización | Dependencia de integración |
| Basada en modelos | Agentes con modelos internos del entorno y decisiones bajo incertidumbre | Entornos dinámicos, información incompleta, planificación adaptativa | Alta | Alto | Flexibilidad y mejor decisión contextual | Mayor complejidad técnica |

### Guía rápida de selección

- Si priorizas control y trazabilidad: usa colaboración basada en reglas.
- Si priorizas productividad por especialización: usa colaboración basada en roles.
- Si priorizas adaptación en escenarios inciertos: usa colaboración basada en modelos.

### Mapeo a este proyecto

- `crewAI` (sección call center) se alinea principalmente con colaboración basada en roles.
- `Ollama tool calling` combina reglas operativas y especialización por herramientas.
- Flujos con razonamiento contextual avanzado pueden evolucionar hacia colaboración basada en modelos.

## Sección adicional: Metodología ReWOO (Planner + Worker + Solver)

En esta sección ejecutarás una implementación ReWOO para resumen de contenido usando:

- Planner: divide la tarea en subpreguntas.
- Worker: usa búsqueda web (Serper) para reunir contexto.
- Solver: sintetiza resultados en una respuesta final.

La implementación ya está creada en `rewoo-summarizer/`.

In [ ]:
# Setup (ejecutar una sola vez)
# %pip install -r rewoo-summarizer/requirements-rewoo.txt

from pathlib import Path
import os
import subprocess

REWOO_BASE = Path("rewoo-summarizer")
assert REWOO_BASE.exists(), "No existe la carpeta rewoo-summarizer"
print("Proyecto ReWOO encontrado en:", REWOO_BASE.resolve())

In [ ]:
# Cargar SERPER_API_KEY desde .env (raiz) o rewoo-summarizer/.env
from pathlib import Path

for env_path in [Path(".env"), REWOO_BASE / ".env"]:
    if env_path.exists():
        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            if key and value and key not in os.environ:
                os.environ[key] = value

if os.getenv("SERPER_API_KEY"):
    print("SERPER_API_KEY detectada.")
else:
    print("Falta SERPER_API_KEY. Definela en .env o rewoo-summarizer/.env.")

In [ ]:
# Ejecutar ReWOO con una tarea concreta
TASK = "Summarize the novella The Metamorphosis"

if not os.getenv("SERPER_API_KEY"):
    print("No se puede ejecutar: falta SERPER_API_KEY")
else:
    proc = subprocess.run(
        ["python", "rewoo_pipeline.py", "--task", TASK],
        cwd=str(REWOO_BASE),
        capture_output=True,
        text=True,
    )
    print("Return code:", proc.returncode)
    print("\n--- STDOUT ---\n")
    print(proc.stdout)
    if proc.stderr:
        print("\n--- STDERR ---\n")
        print(proc.stderr)

In [ ]:
# Smoke test no interactivo de ReWOO
if not os.getenv("SERPER_API_KEY"):
    print("No se puede ejecutar smoke test: falta SERPER_API_KEY")
else:
    proc = subprocess.run(
        ["python", "smoke_test_rewoo.py"],
        cwd=str(REWOO_BASE),
        capture_output=True,
        text=True,
    )
    print("Return code:", proc.returncode)
    print("\n--- STDOUT ---\n")
    print(proc.stdout)
    if proc.stderr:
        print("\n--- STDERR ---\n")
        print(proc.stderr)

### Notas de ejecución ReWOO

- Si no tienes GPU, el pipeline puede tardar más en CPU.
- Si obtienes error HTTP de Serper, revisa `SERPER_API_KEY` y límites de tu plan.
- Puedes cambiar el modelo con `REWOO_MODEL_ID` en `.env`.
- Para tareas más largas, ajusta `REWOO_MAX_EXPERT_LOOPS` y `REWOO_MAX_SUMMARY_LOOPS`.

### Selector interactivo de tarea ReWOO

Usa este selector para lanzar tareas de resumen sin editar manualmente la celda de ejecución.

In [ ]:
# Instalar ipywidgets si no está disponible
import sys
import subprocess

try:
    import ipywidgets  # noqa: F401
    print("ipywidgets ya está instalado.")
except Exception:
    print("Instalando ipywidgets...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets"])
    print("ipywidgets instalado. Si no aparece el widget, reinicia el kernel y vuelve a ejecutar.")

In [ ]:
# Selector interactivo de tarea con ipywidgets (si está disponible)
TASK_OPTIONS = [
    "Summarize the novella The Metamorphosis",
    "Summarize the novel 1984",
    "Summarize the story The Tell-Tale Heart",
    "Summarize the novel Pride and Prejudice",
]

selected_task = TASK_OPTIONS[0]

try:
    import ipywidgets as widgets
    from IPython.display import display

    dropdown = widgets.Dropdown(
        options=TASK_OPTIONS,
        value=TASK_OPTIONS[0],
        description="Task:",
        layout=widgets.Layout(width="80%"),
    )

    run_btn = widgets.Button(description="Run ReWOO", button_style="primary")
    output_box = widgets.Output()

    def _run_selected(_):
        global selected_task
        selected_task = dropdown.value
        output_box.clear_output()
        with output_box:
            if not os.getenv("SERPER_API_KEY"):
                print("No se puede ejecutar: falta SERPER_API_KEY")
                return

            proc = subprocess.run(
                ["python", "rewoo_pipeline.py", "--task", selected_task],
                cwd=str(REWOO_BASE),
                capture_output=True,
                text=True,
            )
            print("Return code:", proc.returncode)
            print("\n--- STDOUT ---\n")
            print(proc.stdout)
            if proc.stderr:
                print("\n--- STDERR ---\n")
                print(proc.stderr)

    run_btn.on_click(_run_selected)
    display(dropdown, run_btn, output_box)

except Exception:
    print("ipywidgets no disponible. Usa selected_task en la celda siguiente.")
    print("selected_task =", selected_task)

In [ ]:
# Fallback por si no usas widget: ejecuta con la variable selected_task
if "selected_task" not in globals():
    selected_task = "Summarize the novella The Metamorphosis"

print("Tarea seleccionada:", selected_task)

if not os.getenv("SERPER_API_KEY"):
    print("No se puede ejecutar: falta SERPER_API_KEY")
else:
    proc = subprocess.run(
        ["python", "rewoo_pipeline.py", "--task", selected_task],
        cwd=str(REWOO_BASE),
        capture_output=True,
        text=True,
    )
    print("Return code:", proc.returncode)
    print("\n--- STDOUT ---\n")
    print(proc.stdout)
    if proc.stderr:
        print("\n--- STDERR ---\n")
        print(proc.stderr)

Si quieres más opciones en el selector, solo agrega nuevos prompts a `TASK_OPTIONS`.